# 02A COF Structures and CIF Files

> 🟢 **Level A · Required**

01B used a ready-made table. This chapter asks where material features originate: unit cells, coordinates, periodicity and CIF parsing.


## COF context: a pore drawing is not a unit cell

In a 2D COF, building blocks form a covalent network within each layer; the 3D crystal also includes the arrangement of those layers. A 3D COF has covalent connectivity extending in three spatial directions. A pore outline does not specify the unit cell: lattice vectors, atomic coordinates and periodic images are needed to reconstruct the structure.

Revisit the [background figure](https://raw.githubusercontent.com/Wanteen/COF-ML-Tutorial/main/assets/cof_background.jpg) and distinguish the molecular units, pore and layer arrangement. The four-atom structure constructed below is a toy example for learning the API, not a chemically complete COF or a model of the structures in the figure. Use a checked CIF for real materials analysis.


## 1. What defines a unit cell?
A general unit cell is described by six lattice parameters:
- `a, b, c`: the three lattice lengths;
- `α, β, γ`: the three lattice angles.

**COFs do not require `a=b`.** Some hexagonal/trigonal structures have `a=b`, but rectangular, orthorhombic, monoclinic or optimized COFs can have `a ≠ b`. We therefore learn the general cell first and treat the hexagonal cell as a special case.


In [ ]:
!pip -q install pymatgen


In [ ]:
from pymatgen.core import Lattice, Structure

# General unit cell: a, b and c can differ and the angles need not be 90 degrees
lattice = Lattice.from_parameters(
    a=18.0, b=22.0, c=3.6,
    alpha=90, beta=90, gamma=95
)
print(lattice)
print('a, b, c =', lattice.a, lattice.b, lattice.c)
print('alpha, beta, gamma =', lattice.alpha, lattice.beta, lattice.gamma)
print('volume =', lattice.volume)


## 2. Hexagonal is a special case
A conventional hexagonal cell usually has `a=b` and `γ=120°`. pymatgen provides a convenient constructor:


In [ ]:
hex_lattice = Lattice.hexagonal(a=12.0, c=3.5)
print(hex_lattice)
print('a =',hex_lattice.a,'b =',hex_lattice.b,'gamma =',hex_lattice.gamma)


## 3. Where are the atoms?
Periodic structures often use **fractional coordinates**. For example, `[0.5, 0.5, 0.5]` means halfway along each of the three lattice vectors.

Fractional coordinates are not measured in Å. They are relative coordinates inside the unit cell. pymatgen can convert them to Cartesian coordinates in Å.


In [ ]:
structure = Structure(
    lattice,
    ['C','C','N','N'],
    [[0,0,0.5],[0.5,0.5,0.5],[0.25,0.25,0.5],[0.75,0.75,0.5]]
)
print('Formula:',structure.composition.reduced_formula)
print('Number of atoms:',len(structure))
print('Volume (A^3):',structure.volume)
print('Density:',structure.density)
for i,site in enumerate(structure):
    print(i,site.species_string,'frac=',site.frac_coords,'cart=',site.coords)


## 4. What does periodicity mean?
The unit cell repeats in space. An atom near the right boundary of the displayed cell may be close to an atom in a neighboring periodic image.

This is the idea of **periodic boundary conditions (PBC)**. Neighbor analysis must therefore consider periodic images, not only the atoms drawn inside one box.


In [ ]:
center=structure[0]
neighbors=structure.get_neighbors(center,r=8.0)
for n in neighbors[:10]:
    print(n.species_string,'distance=',round(n.nn_distance,3),'image=',n.image)


## 5. In research we usually read a CIF directly
```python
from pymatgen.core import Structure
s = Structure.from_file('your_cof.cif')
```

After loading a structure, check:
- `a, b, c, α, β, γ`;
- atom count and composition;
- whether guest molecules or solvent are present;
- whether H atoms are missing;
- disorder / occupancy;
- for 2D COFs, whether interlayer spacing and stacking are reasonable.


## Glossary
- lattice: periodic basis vectors;
- unit cell: smallest repeated structural cell;
- fractional coordinates: coordinates relative to lattice vectors;
- Cartesian coordinates: ordinary x/y/z coordinates, usually in Å;
- PBC: periodic boundary conditions;
- CIF: a common crystallographic file format.


## Exercises
1. Change `a=18` to 20 and inspect the volume.
2. Change `b=22` to 18 and compare `a=b` with `a≠b`.
3. Change `gamma=95` to 90 and inspect the volume.
4. Explain why `Lattice.hexagonal()` cannot represent every COF.
5. Explain the meaning of fractional coordinate `[0.5,0.5,0.5]`.

### Minimum requirement
Know that a general unit cell has six lattice parameters, that `a` and `b` need not be equal, and that a CIF is ultimately parsed into lattice + atoms + coordinates.


## Check your coordinate interpretation
The four-atom cell is an API example, not a chemically complete COF. Changing the cell changes Cartesian positions even when fractional coordinates remain fixed. A supercell changes extensive quantities, while density stays invariant.


In [ ]:
import numpy as np
assert np.allclose(structure.cart_coords, structure.frac_coords @ structure.lattice.matrix)
supercell = structure.copy()
supercell.make_supercell([2, 2, 1])
print('atoms / volume ratios:', len(supercell)/len(structure), supercell.volume/structure.volume)
assert np.isclose(float(supercell.density), float(structure.density))


## Sources and further reading
[Dataset contracts / 数据使用约定](../../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
